In [2]:
"""
将 CB FitAll 与 CB Dynamic 的所有 fea_ 特征 SHAP 排名输出为矩阵文件。

输出文件说明：
1. 每个 trait 输出一个矩阵文件：
   {trait}_ALL_features_rank_matrix.tsv

   - 每一行是一个模型：
     CB_FitAll 或 CB_Dynamic(year_location)
   - 特征列顺序严格按照 CB_FitAll 的 SHAP 排名
   - 单元格中的值是该模型中该特征的 SHAP rank

2. 每个 trait 输出一个 CB_FitAll 特征排序文件：
   {trait}_CB_FitAll_ALL_features_order.tsv

3. 额外输出一个总文件：
   ALL_traits_ALL_features_rank_matrix.tsv

   该文件按 trait 分块保存所有矩阵，保留每个 trait 自己的 CB_FitAll 特征列顺序。
"""

import os
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool


# =========================================================
# 基本参数
# =========================================================
IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

OUTDIR =  "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/"
os.makedirs(OUTDIR, exist_ok=True)

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

cat_features = ["location", "year"]
min_env_samples = 5

catboost_params = {
    "iterations": 1000,
    "thread_count": 48,
    "verbose": 0,
    "random_seed": 42
}

In [3]:

# =========================================================
# 函数1：规范 year，去掉 .0
# =========================================================
def clean_year_value(y):
    return str(int(float(y)))


# =========================================================
# 函数2：计算 SHAP 排名
# =========================================================
def get_shap_rank_df(model, data_df, feature_cols, cat_features, keep_only_fea=True):
    pool = Pool(
        data=data_df[feature_cols],
        cat_features=cat_features
    )

    shap_values = model.get_feature_importance(pool, type="ShapValues")
    shap_values = np.array(shap_values)

    # CatBoost SHAP 最后一列是 expected value
    shap_values = shap_values[:, :-1]

    shap_df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0)
    })

    if keep_only_fea:
        shap_df = shap_df[shap_df["feature"].str.startswith("fea_")].copy()

    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_df["rank"] = np.arange(1, len(shap_df) + 1)

    return shap_df


# =========================================================
# 函数3：根据指定特征顺序提取 rank
# =========================================================
def get_rank_values(shap_rank_df, ordered_features):
    rank_map = dict(zip(shap_rank_df["feature"], shap_rank_df["rank"]))
    return [rank_map.get(feature, np.nan) for feature in ordered_features]


# =========================================================
# 主程序
# =========================================================
all_trait_matrices = []

for target_col in traits:

    print(f"\n========== Running trait: {target_col} ==========")

    train_file = os.path.join(IN_PATH, f"train-{target_col}.txt")
    if not os.path.exists(train_file):
        print(f"Skip {target_col}: file not found -> {train_file}")
        continue

    train_df = pd.read_table(train_file)

    train_df["location"] = train_df["location"].astype(str)
    train_df["year"] = train_df["year"].apply(clean_year_value)

    fea_cols = [c for c in train_df.columns if c.startswith("fea_")]
    model_feature_cols = fea_cols + cat_features

    # year × location 组合，环境变量仍然使用 location/year
    pairs = (
        train_df[["year", "location"]]
        .drop_duplicates()
        .sort_values(["year", "location"])
    )
    pairs = list(pairs.itertuples(index=False, name=None))

    # =====================================================
    # CB_FitAll：使用全部训练数据训练
    # =====================================================
    whole_pool = Pool(
        data=train_df[model_feature_cols],
        label=train_df[target_col],
        cat_features=cat_features
    )

    whole_model = CatBoostRegressor(**catboost_params)
    whole_model.fit(whole_pool)

    whole_shap_rank = get_shap_rank_df(
        model=whole_model,
        data_df=train_df,
        feature_cols=model_feature_cols,
        cat_features=cat_features,
        keep_only_fea=True
    )

    # 所有特征按照 CB_FitAll 的 SHAP 排名排序
    ordered_features = whole_shap_rank["feature"].tolist()
    ordered_feature_names = [
        feature.replace("fea_", "", 1) for feature in ordered_features
    ]

    # 保存 CB_FitAll 的完整特征排序
    whole_order_df = whole_shap_rank.copy()
    whole_order_df.insert(0, "Trait", target_col)
    whole_order_df["feature_display"] = whole_order_df["feature"].str.replace(
        "^fea_", "", regex=True
    )

    whole_order_tsv = os.path.join(
        OUTDIR,
        f"{target_col}_CB_FitAll_ALL_features_order.tsv"
    )
    whole_order_df.to_csv(whole_order_tsv, sep="\t", index=False)

    # =====================================================
    # 构建 rank matrix
    # 第一行是 CB_FitAll，后续行是每个 CB_Dynamic(year_location)
    # =====================================================
    matrix_rows = []

    matrix_rows.append(
        ["CB_FitAll", "ALL", "ALL"] + list(range(1, len(ordered_features) + 1))
    )

    for year, location in pairs:

        year_clean = clean_year_value(year)

        train_data = train_df[
            (train_df["year"] == year_clean) &
            (train_df["location"] == location)
        ].copy()

        if len(train_data) < min_env_samples:
            print(f"Skip {year_clean}_{location}: too few samples ({len(train_data)})")
            continue

        train_pool = Pool(
            data=train_data[model_feature_cols],
            label=train_data[target_col],
            cat_features=cat_features
        )

        # CB_Dynamic：以 CB_FitAll 为初始模型，在当前 year/location 环境中继续训练
        dynamic_model = CatBoostRegressor(**catboost_params)
        dynamic_model.fit(train_pool, init_model=whole_model)

        dynamic_shap_rank = get_shap_rank_df(
            model=dynamic_model,
            data_df=train_data,
            feature_cols=model_feature_cols,
            cat_features=cat_features,
            keep_only_fea=True
        )

        dynamic_rank_values = get_rank_values(
            shap_rank_df=dynamic_shap_rank,
            ordered_features=ordered_features
        )

        matrix_rows.append(
            [f"CB_Dynamic({year_clean}_{location})", int(year_clean), location]
            + dynamic_rank_values
        )

    rank_matrix_df = pd.DataFrame(
        matrix_rows,
        columns=["model_name", "year", "location"] + ordered_feature_names
    )

    # =====================================================
    # 输出当前 trait 的完整 rank matrix
    # =====================================================
    rank_matrix_tsv = os.path.join(
        OUTDIR,
        f"{target_col}_ALL_features_rank_matrix.tsv"
    )
    rank_matrix_df.to_csv(rank_matrix_tsv, sep="\t", index=False)

    all_trait_matrices.append((target_col, rank_matrix_df))

    print(f"Saved rank matrix : {rank_matrix_tsv}")
    print(f"Saved FitAll order: {whole_order_tsv}")


# =========================================================
# 汇总输出：一个文件中按 trait 分块保存所有 rank matrix
# =========================================================
combined_tsv = os.path.join(
    OUTDIR,
    "ALL_traits_ALL_features_rank_matrix.tsv"
)

if len(all_trait_matrices) > 0:
    with open(combined_tsv, "w", encoding="utf-8") as f:
        for target_col, rank_matrix_df in all_trait_matrices:
            f.write(f"# Trait\t{target_col}\n")
            rank_matrix_df.to_csv(f, sep="\t", index=False)
            f.write("\n")

    print(f"\nSaved combined matrix: {combined_tsv}")

print("\nAll done.")


========== Running trait: Yield_per_plant ==========
Saved rank matrix : /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/Yield_per_plant_ALL_features_rank_matrix.tsv
Saved FitAll order: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/Yield_per_plant_CB_FitAll_ALL_features_order.tsv

========== Running trait: 1000-grain_weight ==========
Saved rank matrix : /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/1000-grain_weight_ALL_features_rank_matrix.tsv
Saved FitAll order: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/1000-grain_weight_CB_FitAll_ALL_features_order.tsv

========== Running trait: Grain_length_to_width_ratio ==========
Saved rank matrix : /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/Grain_length_to_width_ratio_ALL_features_rank_matrix.tsv
Saved FitAll order: /data2/zhoujb/project/cro